# Proyecto final: E2D2 para traduccion de secuencias

#### Integrantes:
1. Nicolás Vásquez Renjifo
2. Jorge Luis Fong Gutierrez
3. Jhonatan David Rengifo
4. Mateo González Ruiz

Este notebook documenta la implementacion experimental de **E2D2: Encoder-Decoder Diffusion Language Models**, una arquitectura Transformer encoder-decoder para modelos de difusion discreta aplicada a traduccion automatica aleman -> ingles.

El objetivo no es entrenar un modelo desde cero. La idea es comprender una arquitectura Transformer encoder-decoder, usar un articulo cientifico con codigo y pesos preentrenados, implementar inferencia y explicar su funcionamiento en profundidad. Por eso este notebook combina:

1. Carga e inferencia con pesos preentrenados oficiales.
2. Inspeccion interna de la arquitectura.
3. Ablation study sobre parametros de generacion de E2D2.
4. Comparacion contra un baseline Transformer encoder-decoder autoregresivo.
5. Analisis cualitativo de errores.

Fuentes principales:
- Paper: https://arxiv.org/abs/2510.22852
- Repositorio oficial: https://github.com/kuleshov-group/e2d2
- Modelo WMT preentrenado: https://huggingface.co/kuleshov-group/e2d2-wmt
- Coleccion de modelos: https://huggingface.co/collections/kuleshov-group/e2d2


## 1. Decision experimental

El modelo principal es `kuleshov-group/e2d2-wmt`, entrenado para traduccion WMT14 de-en. Esta eleccion cumple la consigna porque la tarea usa lenguaje natural, una secuencia compleja no tabular, y porque E2D2 separa explicitamente el procesamiento en un **encoder grande** y un **decoder de difusion**.

Configuracion principal:
- Modelo: `kuleshov-group/e2d2-wmt`
- Tarea: traduccion aleman -> ingles
- Dataset de prueba: WMT14 de-en
- Baseline comparativo: `Helsinki-NLP/opus-mt-de-en`
- Metricas: BLEU, chrF, tiempo promedio, tokens/s y repeticion
- Entorno: Google Colab con GPU Tesla T4 en la corrida registrada

La configuracion oficial del checkpoint E2D2-WMT usa `block_size=4`, `num_steps=4` y cache activado. Esa configuracion se toma como baseline interno de E2D2.


## 1.1. Cumplimiento explícito de la consigna

| Requisito de la consigna | Cómo se cumple en este proyecto |
|---|---|
| Arquitectura Transformer encoder-decoder | E2D2 usa un encoder Transformer grande y un decoder Transformer de difusión. |
| Datos secuenciales complejos | Se trabaja con lenguaje natural en traducción alemán -> inglés. |
| No usar series de tiempo tradicionales | La tarea no usa datos tabulares, sensores ni registros estructurados. |
| Artículo científico base | `Encoder-Decoder Diffusion Language Models for Efficient Training and Inference`. |
| Código disponible | Repositorio oficial `https://github.com/kuleshov-group/e2d2`. |
| Pesos preentrenados | Checkpoint `kuleshov-group/e2d2-wmt` en Hugging Face. |
| No entrenar desde cero | Solo se implementa inferencia con pesos preentrenados. |
| Explicar funcionamiento | El notebook incluye arquitectura, máscaras conceptuales, block diffusion, métricas y análisis cualitativo. |
| Evaluación experimental | Se comparan configuraciones E2D2 y un baseline OPUS-MT sobre WMT14 de-en. |


## 1.2. Diagrama conceptual de E2D2

El flujo de inferencia usado por E2D2 puede entenderse así:

```text
Texto fuente en alemán
        |
        v
+-------------------------------+
| Encoder Transformer grande    |
| Representa tokens limpios     |
| 28 capas en E2D2-WMT          |
+-------------------------------+
        |
        | representaciones h
        v
+-------------------------------+
| Decoder Transformer liviano   |
| Denoising de bloque activo    |
| 4 capas en E2D2-WMT           |
+-------------------------------+
        ^
        |
        | T pasos de difusión por bloque
        |
[MASK/MASK/MASK/MASK] -> tokens refinados -> bloque generado
        |
        v
Traducción en inglés
```

La idea central es que el encoder procesa contexto limpio y el decoder refina tokens ruidosos por bloques. Esto diferencia a E2D2 de un encoder-decoder autoregresivo tradicional como OPUS-MT.


## 2. Verificacion de acelerador en Colab

Se verificó el acelerador en Colab desde Runtime > Change runtime type > GPU. Se seleccionó una T4, suficiente para correr el modelo WMT con muestras pequeñas.

En cuanto a la TPU v5e-1, se identificó que no utiliza CUDA, por lo que se trató como opción experimental. El notebook tiene la capacidad de detectar TPU si torch_xla está disponible, pero dado que E2D2 depende de código custom de PyTorch/Transformers orientado a CUDA, se priorizó la ejecución en GPU para evitar incompatibilidades.

In [1]:
# Importa PyTorch, base de ejecucion del modelo y tensores.
import torch

print('Torch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
# Ruta preferida: ejecucion con GPU CUDA en Colab.
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM total GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    try:
        import torch_xla.core.xla_model as xm
        print('TPU/XLA disponible:', xm.xla_device())
    except Exception as exc:
        print('TPU/XLA no detectado:', repr(exc))


Torch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4
VRAM total GB: 15.64


## 3. Instalacion de dependencias

El repositorio oficial usa `transformers==4.52.4`, `datasets==4.2.0` y `accelerate==1.11.0`. En Colab instalamos las piezas necesarias para inferencia y evaluacion pequena.

In [2]:
!pip -q install \
  "transformers==4.52.4" \
  "accelerate==1.11.0" \
  "datasets==4.2.0" \
  "evaluate" \
  "sacrebleu" \
  "sentencepiece==0.2.1" \
  "einops==0.8.1" \
  "hydra-core==1.3.2"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 70.7 MB/s eta 0:00:00


Despues de instalar, Colab solicita de vez en cuando reiniciar el runtime y continuar desde la siguiente celda.

In [3]:
# Permite clonar configuraciones sin modificar los objetos originales.
import copy
# Permite medir tiempo de inferencia.
import time
# Organiza resultados experimentales en tablas.
import pandas as pd
# Importa PyTorch, base de ejecucion del modelo y tensores.
import torch
# Descarga datasets desde Hugging Face.
from datasets import load_dataset
# Carga metricas como BLEU y chrF.
import evaluate
# Carga el modelo E2D2 y su tokenizador mediante la API de Transformers.
from transformers import AutoModelForMaskedLM, AutoTokenizer
# Define un criterio de parada cuando aparece el token EOS.
from transformers.generation.stopping_criteria import EosTokenCriteria

# Checkpoint oficial usado como modelo principal.
MODEL_ID = 'kuleshov-group/e2d2-wmt'

# Revision exacta del checkpoint E2D2-WMT para reproducibilidad.
MODEL_REVISION = '7c65309acc0d6d89049c9aca4ea02f9d10d1da5a'

# Bandera para distinguir TPU/XLA de CUDA.
USING_XLA = False
# Ruta preferida: ejecucion con GPU CUDA en Colab.
if torch.cuda.is_available():
    # Selecciona GPU como dispositivo de computo.
    DEVICE = torch.device('cuda')
    # Usa precision float32 para estabilidad en este checkpoint.
    DTYPE = torch.float32
else:
    try:
        import torch_xla.core.xla_model as xm
        # Selecciona TPU cuando XLA esta disponible.
        DEVICE = xm.xla_device()
        # bfloat16 es el tipo numerico natural para TPU.
        DTYPE = torch.bfloat16
        USING_XLA = True
    except Exception:
        # Fallback a CPU si no hay acelerador.
        DEVICE = torch.device('cpu')
        # Usa precision float32 para estabilidad en este checkpoint.
        DTYPE = torch.float32

print('Device:', DEVICE)
print('Dtype:', DTYPE)
print('Usando XLA/TPU:', USING_XLA)


Device: cuda
Dtype: torch.float32
Usando XLA/TPU: False


## 4. Carga del modelo preentrenado

`trust_remote_code=True` es necesario porque Hugging Face descarga las clases custom de E2D2: configuracion de difusion, backbone encoder-decoder y metodo `generate`.

In [4]:
# Carga el tokenizador publicado junto al checkpoint E2D2.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, revision=MODEL_REVISION)

# Carga E2D2 como denoiser de difusion discreta.
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_ID,
    # Requerido porque E2D2 publica clases custom en Hugging Face.
    trust_remote_code=True,
    torch_dtype=DTYPE,
    attn_implementation='eager',
    low_cpu_mem_usage=True,
).to(DEVICE)
# Activa modo evaluacion para inferencia.
model.eval()

# Cuenta parametros totales del modelo.
n_params = sum(p.numel() for p in model.parameters())
print(f'Modelo: {MODEL_ID}')
print(f'Parametros: {n_params/1e6:.1f}M')
print('Generation config:', model.generation_config)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Some weights of E2D2 were not initialized from the model checkpoint at kuleshov-group/e2d2-wmt and are newly initialized: ['encoder_static_attention_mask', 'static_attention_mask']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/427 [00:00<?, ?B/s]

Modelo: kuleshov-group/e2d2-wmt
Parametros: 331.8M
Generation config: GenerationConfig {
  "align_inputs_to_blocks": false,
  "block_size": 4,
  "bos_token_id": 151643,
  "confidence_based_noising": true,
  "confidence_margin_based_noising": false,
  "confidence_threshold": 1000000.0,
  "eos_token_id": 151643,
  "first_hitting": true,
  "min_t": 1e-05,
  "num_steps": 4,
  "pad_token_id": 151643,
  "sampling_strategy": "predict_and_noise",
  "use_model_output_cache": true
}



## 4.1. Reproducibilidad de checkpoints

Para evitar que una actualización remota cambie silenciosamente el resultado, el notebook fija las revisiones de Hugging Face:

- E2D2-WMT: `7c65309acc0d6d89049c9aca4ea02f9d10d1da5a`
- OPUS-MT de-en: `1a922f3b32a8e809e17a47d4b32142d8105924e5`

Si se desea usar la versión más reciente, se puede retirar el argumento `revision`.


## 5. Funcion de traduccion

El dataset oficial construye la entrada como: `BOS + texto_aleman + EOS`. El modelo genera la continuacion, que corresponde a la traduccion en ingles.

In [ ]:
# Se Construye el prompt de traduccion con BOS/EOS como en el repo oficial.
def build_source_text(german_text: str) -> str:
    bos = tokenizer.bos_token or tokenizer.eos_token or ''
    eos = tokenizer.eos_token or ''
    return bos + german_text.strip() + eos


# Funcion principal de inferencia E2D2.
def translate_e2d2(
    german_text: str,
    max_new_tokens: int = 64,
    block_size: int = 4,
    num_steps: int = 4,
    use_cache: bool = True,
):
    # Tokeniza la frase alemana y mueve tensores al acelerador.
    inputs = tokenizer(build_source_text(german_text), return_tensors='pt').to(DEVICE)

    # Copia la configuracion de generacion para modificarla por experimento.
    gen_config = copy.deepcopy(model.generation_config)
    # Define cuantos tokens se refinan por bloque.
    gen_config.block_size = block_size
    # Define cuantos pasos de denoising se hacen por bloque.
    gen_config.num_steps = num_steps
    # Activa/desactiva cache de generacion.
    gen_config.use_cache = use_cache
    gen_config.use_model_output_cache = use_cache

    # Ruta preferida: ejecucion con GPU CUDA en Colab.
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    # Inicio de medicion de tiempo.
    t0 = time.perf_counter()
    # Desactiva gradientes porque solo se hace inferencia.
    with torch.inference_mode():
        # Ejecuta el sampling/generacion del modelo.
        output_ids = model.generate(
            inputs=inputs['input_ids'],
            max_new_tokens=max_new_tokens,
            generation_config=gen_config,
            stopping_criteria=EosTokenCriteria(tokenizer.eos_token_id),
            disable_pbar=True,
        )
    if USING_XLA:
        try:
            import torch_xla.core.xla_model as xm
            xm.mark_step()
        except Exception:
            pass
    # Ruta preferida: ejecucion con GPU CUDA en Colab.
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    # Conserva solo los tokens generados, no el texto fuente.
    new_ids = output_ids[0, inputs['input_ids'].shape[-1]:]
    # Convierte tokens generados a texto limpio.
    decoded = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    # Cuenta tokens generados para throughput.
    generated_tokens = int(new_ids.numel())
    return {
        'source_de': german_text,
        'prediction_en': decoded,
        'seconds': elapsed,
        'generated_tokens': generated_tokens,
        'tokens_per_second': generated_tokens / max(elapsed, 1e-9),
        'block_size': block_size,
        'num_steps': num_steps,
        'use_cache': use_cache,
    }


## 6. Prueba manual de inferencia

Estas frases permiten revisar rapidamente si el modelo esta generando traducciones coherentes antes de pasar al dataset.

In [6]:
# Frases de prueba manual para validar inferencia.
manual_examples = [
    'Guten Morgen, wie geht es Ihnen heute?',
    'Die Wissenschaftler entwickelten ein neues Modell fuer maschinelle Uebersetzung.',
    'Das Wetter ist heute kalt, aber der Himmel ist klar.',
]

# Traduce los ejemplos manuales con la configuracion oficial.
manual_results = [translate_e2d2(x, max_new_tokens=64) for x in manual_examples]
pd.DataFrame(manual_results)


,source_de,prediction_en,seconds,generated_tokens,tokens_per_second,block_size,num_steps,use_cache
0,"Guten Morgen, wie geht es Ihnen heute?","Good morning, what are you doing today, Commis...",2.692682,12,4.456523,4,4,True
1,Die Wissenschaftler entwickelten ein neues Mod...,The scientists developed a new model for machi...,0.519263,12,23.109694,4,4,True
2,"Das Wetter ist heute kalt, aber der Himmel ist...","The weather is cold today, but the sky is clea...",1.567364,32,20.416448,4,4,True


## 6.1. Traducción interactiva en vivo

Esta sección permite escribir un texto en alemán y traducirlo inmediatamente. Sirve para demostración y validación del proyecto. La primera celda usa un texto fijo editable; la segunda usa widgets interactivos de Colab/Jupyter.


In [7]:
# Texto editable para una prueba rapida en vivo.
live_german_text = 'Die Architektur des Modells trennt die Darstellung sauberer Tokens vom iterativen Entrauschen.'

# Traduce con E2D2 usando la configuracion oficial del checkpoint.
live_e2d2 = translate_e2d2(
    live_german_text,
    max_new_tokens=96,
    block_size=4,
    num_steps=4,
    use_cache=True,
)

# Si el baseline ya esta cargado, tambien traduce con OPUS-MT para comparacion inmediata.
if 'translate_opus' in globals():
    live_opus = translate_opus(live_german_text, max_new_tokens=96)
else:
    live_opus = {'prediction_en': 'OPUS-MT no esta cargado todavia. Ejecuta primero la seccion del baseline.'}

# Muestra los resultados de forma compacta.
pd.DataFrame([
    {
        'modelo': 'E2D2 official_b4_s4_cache',
        'texto_de': live_german_text,
        'traduccion_en': live_e2d2['prediction_en'],
        'segundos': live_e2d2['seconds'],
        'tokens_s': live_e2d2['tokens_per_second'],
    },
    {
        'modelo': 'OPUS autoregressive_encoder_decoder',
        'texto_de': live_german_text,
        'traduccion_en': live_opus['prediction_en'],
        'segundos': live_opus.get('seconds', None),
        'tokens_s': live_opus.get('tokens_per_second', None),
    },
])


,modelo,texto_de,traduccion_en,segundos,tokens_s
0,E2D2 official_b4_s4_cache,Die Architektur des Modells trennt die Darstel...,The architecture of the model separates the re...,1.446197,22.126992
1,OPUS autoregressive_encoder_decoder,Die Architektur des Modells trennt die Darstel...,OPUS-MT no esta cargado todavia. Ejecuta prime...,NaN,NaN


### Widget interactivo


In [20]:
# Widget interactivo para Colab/Jupyter.
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    text_area = widgets.Textarea(
        value='Guten Abend, wir testen eine Live-Übersetzung mit dem E2D2-Modell.',
        placeholder='Escribe una frase en aleman...',
        description='Alemán:',
        layout=widgets.Layout(width='100%', height='90px'),
    )

    max_tokens_slider = widgets.IntSlider(
        value=96,
        min=16,
        max=160,
        step=8,
        description='Max tokens:',
        continuous_update=False,
    )

    compare_checkbox = widgets.Checkbox(
        value=True,
        description='Comparar con OPUS-MT si esta cargado',
    )

    run_button = widgets.Button(
        description='Traducir',
        button_style='primary',
    )

    output_box = widgets.Output()

    def run_live_translation(_):
        with output_box:
            clear_output(wait=True)
            text = text_area.value.strip()
            if not text:
                print('Escribe primero un texto en aleman.')
                return

            e2d2_result = translate_e2d2(
                text,
                max_new_tokens=max_tokens_slider.value,
                block_size=4,
                num_steps=4,
                use_cache=True,
            )

            rows = [{
                'modelo': 'E2D2 official_b4_s4_cache',
                'traduccion_en': e2d2_result['prediction_en'],
                'segundos': e2d2_result['seconds'],
                'tokens_s': e2d2_result['tokens_per_second'],
            }]

            if compare_checkbox.value and 'translate_opus' in globals():
                opus_result = translate_opus(text, max_new_tokens=max_tokens_slider.value)
                rows.append({
                    'modelo': 'OPUS autoregressive_encoder_decoder',
                    'traduccion_en': opus_result['prediction_en'],
                    'segundos': opus_result['seconds'],
                    'tokens_s': opus_result['tokens_per_second'],
                })

            display(pd.DataFrame(rows))

    run_button.on_click(run_live_translation)
    display(text_area, max_tokens_slider, compare_checkbox, run_button, output_box)
except Exception as exc:
    print('No fue posible inicializar widgets interactivos:', repr(exc))
    print('Usa la celda anterior editando la variable live_german_text.')


Textarea(value='Guten Abend, wir testen eine Live-Übersetzung mit dem E2D2-Modell.', description='Alemán:', la…

IntSlider(value=96, continuous_update=False, description='Max tokens:', max=160, min=16, step=8)

Checkbox(value=True, description='Comparar con OPUS-MT si esta cargado')

Button(button_style='primary', description='Traducir', style=ButtonStyle())

Output()

## 7. Carga de muestra WMT14 de-en

Usaremos una muestra pequeña para que el notebook sea viable en Colab. El objetivo no es reproducir la tabla completa del paper, sino demostrar inferencia y evaluacion.

In [9]:
# Muestra rapida para una primera medicion.
N_EVAL = 10
# Descarga una muestra del test set WMT14 de-en.
wmt = load_dataset('wmt/wmt14', 'de-en', split=f'test[:{N_EVAL}]')

# Acumula pares fuente/referencia.
sample_rows = []
for row in wmt:
    sample_rows.append({
        'de': row['translation']['de'],
        'reference_en': row['translation']['en'],
    })

pd.DataFrame(sample_rows).head()


README.md: 0.00B [00:00, ?B/s]

de-en/train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

de-en/train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

de-en/train-00002-of-00003.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

de-en/validation-00000-of-00001.parquet:   0%|          | 0.00/474k [00:00<?, ?B/s]

de-en/test-00000-of-00001.parquet:   0%|          | 0.00/509k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

,de,reference_en
0,Gutach: Noch mehr Sicherheit für Fußgänger,Gutach: Increased safety for pedestrians
1,Sie stehen keine 100 Meter voneinander entfern...,They are not even 100 metres apart: On Tuesday...
2,Zwei Anlagen so nah beieinander: Absicht oder ...,Two sets of lights so close to one another: in...
3,Diese Frage hat Gutachs Bürgermeister gestern ...,"Yesterday, Gutacht's Mayor gave a clear answer..."
4,"""Die Rathausampel ist damals installiert worde...","""At the time, the Town Hall traffic lights wer..."


## 8. Evaluacion pequena con BLEU

BLEU se calcula sobre pocas muestras; por eso debe interpretarse como evidencia funcional, no como reproduccion exacta del paper.

In [10]:
# Acumula predicciones para metricas.
predictions = []
# Acumula referencias humanas.
references = []
# Acumula registros completos de inferencia.
records = []

for row in sample_rows:
    out = translate_e2d2(row['de'], max_new_tokens=96, block_size=4, num_steps=4, use_cache=True)
    predictions.append(out['prediction_en'])
    references.append(row['reference_en'])
    records.append({**row, **out})

results_df = pd.DataFrame(records)
display(results_df[['de', 'reference_en', 'prediction_en', 'seconds', 'tokens_per_second']])

# Carga SacreBLEU.
bleu = evaluate.load('sacrebleu')
# Calcula BLEU sobre predicciones y referencias.
bleu_result = bleu.compute(predictions=predictions, references=[[r] for r in references])
bleu_result


,de,reference_en,prediction_en,seconds,tokens_per_second
0,Gutach: Noch mehr Sicherheit für Fußgänger,Gutach: Increased safety for pedestrians,Well: more safety for pedestrians.,0.263191,30.396153
1,Sie stehen keine 100 Meter voneinander entfern...,They are not even 100 metres apart: On Tuesday...,They are not 100 metres away: On Tuesday the n...,1.046920,42.028032
2,Zwei Anlagen so nah beieinander: Absicht oder ...,Two sets of lights so close to one another: in...,Two plants so close to each other: intention o...,0.291048,54.973671
3,Diese Frage hat Gutachs Bürgermeister gestern ...,"Yesterday, Gutacht's Mayor gave a clear answer...",Gutach Mayor gave a clear answer to this quest...,0.650405,55.350168
4,"""Die Rathausampel ist damals installiert worde...","""At the time, the Town Hall traffic lights wer...","""The Town Hall lamp was installed at that time...",0.428091,56.062850
5,Die Kluser-Ampel sichere sowohl Radfahrer als ...,"The Kluser lights protect cyclists, as well as...",The Kluser lights protect both cyclists and bu...,0.637330,56.485618
6,Die gestern offiziell in Betrieb genommene Anl...,"The system, which officially became operationa...",The facility officially opened yesterday was i...,0.645516,55.769318
7,"Wir haben das Museum, zwei Kirchen, Kurpark, d...","We have the museum, two churches, the spa gard...","We have a museum, two churches, spa park, bus ...",0.998247,36.063221
8,"""Bei dem hohen Verkehrs- und Fußgängeraufkomme...","""At times of high road and pedestrian traffic,...","""The high traffic and pedestrian volume had to...",0.764615,31.388339
9,Dies bestätigt auch Peter Arnold vom Landratsa...,This was also confirmed by Peter Arnold from t...,This is also confirmed by Peter Arnold of the ...,0.274020,58.389951


{'score': 17.244436482625247,
 'counts': [119, 52, 27, 15],
 'totals': [246, 236, 226, 216],
 'precisions': [48.3739837398374,
  22.033898305084747,
  11.946902654867257,
  6.944444444444445],
 'bp': 1.0,
 'sys_len': 246,
 'ref_len': 203}

## 9. Experimentos de parámetros de difusión

La hipotesis del paper es que E2D2 mejora el trade-off calidad/velocidad al usar un encoder grande y un decoder pequeño. En esta seccion variamos `block_size` y `num_steps` para observar el costo de inferencia.

In [11]:
# Usa un ejemplo para exploracion rapida de parametros.
experiment_text = sample_rows[0]['de']
# Grilla exploratoria de block_size y num_steps.
grid = [
    {'block_size': 2, 'num_steps': 2},
    {'block_size': 4, 'num_steps': 4},
    {'block_size': 8, 'num_steps': 4},
]

experiment_results = []
for cfg in grid:
    experiment_results.append(
        translate_e2d2(
            experiment_text,
            max_new_tokens=96,
            block_size=cfg['block_size'],
            num_steps=cfg['num_steps'],
            use_cache=True,
        )
    )

pd.DataFrame(experiment_results)[[
    'block_size', 'num_steps', 'seconds', 'tokens_per_second', 'prediction_en'
]]


,block_size,num_steps,seconds,tokens_per_second,prediction_en
0,2,2,0.052649,37.987372,Well
1,4,4,0.140234,57.047663,Well: more safety for pedestrians.
2,8,4,0.314480,101.755170,Well well: even more safety safety for pedestr...


### Interpretacion de la prueba exploratoria

La prueba con una sola frase muestra que los parametros de difusion no son intercambiables. `block_size=2` genera salidas demasiado cortas; `block_size=8` puede aumentar tokens por segundo, pero tambien introduce repeticion. Por eso se conserva la configuracion oficial `block_size=4`, `num_steps=4`, `use_cache=True` como baseline interno de E2D2.


## 10. Inspeccion interna de la arquitectura

Esta seccion convierte el uso del modelo en analisis arquitectónico. Se verifican los componentes reportados por el paper y por la configuracion del checkpoint: número de capas del encoder y decoder, dimensiones internas, tipo de difusion, longitud máxima y parámetros de block diffusion.


In [12]:
# Cuenta parametros de un modulo PyTorch.
def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

# Extrae configuracion interna del checkpoint.
config_dict = model.config.to_dict()
# Extrae configuracion del backbone encoder-decoder.
backbone_cfg = config_dict.get('backbone_config', {})

architecture_rows = [
    {'atributo': 'model_type', 'valor': config_dict.get('model_type')},
    {'atributo': 'diffusion_type', 'valor': config_dict.get('diffusion_type')},
    {'atributo': 'length', 'valor': config_dict.get('length')},
    {'atributo': 'block_size', 'valor': config_dict.get('block_size')},
    {'atributo': 'eval_block_size', 'valor': config_dict.get('eval_block_size')},
    {'atributo': 'mask_token_id', 'valor': config_dict.get('mask_token_id')},
    {'atributo': 'tokenizer_name', 'valor': config_dict.get('tokenizer_name')},
    {'atributo': 'encoder_layers', 'valor': backbone_cfg.get('num_encoder_layers')},
    {'atributo': 'decoder_layers', 'valor': backbone_cfg.get('num_decoder_layers')},
    {'atributo': 'hidden_size', 'valor': backbone_cfg.get('hidden_size')},
    {'atributo': 'intermediate_size', 'valor': backbone_cfg.get('intermediate_size')},
    {'atributo': 'backbone_class', 'valor': backbone_cfg.get('_target_')},
    {'atributo': 'tie_encoder_decoder_weights', 'valor': backbone_cfg.get('tie_encoder_decoder_weights')},
]

# Tabla de atributos arquitectonicos.
architecture_df = pd.DataFrame(architecture_rows)
display(architecture_df)

children_rows = []
for name, child in model.named_children():
    children_rows.append({
        'modulo': name,
        'clase': type(child).__name__,
        'parametros_millones': count_parameters(child) / 1e6,
    })

children_df = pd.DataFrame(children_rows)
display(children_df)

print(f'Parametros totales: {count_parameters(model)/1e6:.1f}M')


,atributo,valor
0,model_type,e2d2
1,diffusion_type,absorbing
2,length,256
3,block_size,4
4,eval_block_size,4
5,mask_token_id,151660
6,tokenizer_name,Qwen/Qwen3-0.6B-Base
7,encoder_layers,28
8,decoder_layers,4
9,hidden_size,512


,modulo,clase,parametros_millones
0,backbone,LLMasEncoderDecoder,331.785216


Parametros totales: 331.8M


## 11. Visualizacion conceptual de block diffusion

E2D2 usa block diffusion: la secuencia objetivo se genera/refina por bloques. El encoder mantiene representaciones de tokens limpios/contexto, mientras el decoder hace denoising iterativo de los tokens activos. Esta celda ilustra la asignacion de tokens a bloques y las mascaras conceptuales descritas en el paper.


In [13]:
# Ilustra asignacion de tokens a bloques.
def block_table(seq_len=16, block_size=4):
    return pd.DataFrame({
        'posicion_token': list(range(seq_len)),
        'bloque': [i // block_size for i in range(seq_len)],
        'token_simbolico': [f'x_{i}' for i in range(seq_len)],
    })


# Construye mascara conceptual block-causal del encoder.
def encoder_mask_table(seq_len=16, block_size=4):
    data = []
    for q in range(seq_len):
        row = []
        for kv in range(seq_len):
            row.append(1 if (q // block_size) >= (kv // block_size) else 0)
        data.append(row)
    return pd.DataFrame(data, index=[f'q{q}' for q in range(seq_len)], columns=[f'k{kv}' for kv in range(seq_len)])


# Construye mascara conceptual del decoder.
def decoder_mask_table(seq_len=16, block_size=4):
    # Columnas h_* representan salidas limpias del encoder; z_* representa tokens ruidosos del decoder.
    columns = [f'h{kv}' for kv in range(seq_len)] + [f'z{kv}' for kv in range(seq_len)]
    data = []
    for q in range(seq_len):
        row = []
        q_block = q // block_size
        for kv in range(2 * seq_len):
            if kv < seq_len:
                kv_block = kv // block_size
                attends = q_block > kv_block
            else:
                z_idx = kv - seq_len
                kv_block = z_idx // block_size
                attends = q_block == kv_block
            row.append(1 if attends else 0)
        data.append(row)
    return pd.DataFrame(data, index=[f'q{q}' for q in range(seq_len)], columns=columns)

print('Asignacion de tokens a bloques')
display(block_table(seq_len=16, block_size=4))

print('Mascara conceptual del encoder: block-causal')
display(encoder_mask_table(seq_len=16, block_size=4))

print('Mascara conceptual del decoder: cross-attention a bloques previos + self-attention dentro del bloque activo')
display(decoder_mask_table(seq_len=16, block_size=4))


Asignacion de tokens a bloques


,posicion_token,bloque,token_simbolico
0,0,0,x_0
1,1,0,x_1
2,2,0,x_2
3,3,0,x_3
4,4,1,x_4
5,5,1,x_5
6,6,1,x_6
7,7,1,x_7
8,8,2,x_8
9,9,2,x_9


Mascara conceptual del encoder: block-causal


,k0,k1,k2,k3,k4,k5,k6,k7,k8,k9,k10,k11,k12,k13,k14,k15
q0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
q1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
q2,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
q3,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
q4,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0
q5,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0
q6,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0
q7,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0
q8,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0
q9,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0


Mascara conceptual del decoder: cross-attention a bloques previos + self-attention dentro del bloque activo


,h0,h1,h2,h3,h4,h5,h6,h7,h8,h9,...,z6,z7,z8,z9,z10,z11,z12,z13,z14,z15
q0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
q1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
q2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
q3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
q4,1,1,1,1,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
q5,1,1,1,1,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
q6,1,1,1,1,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
q7,1,1,1,1,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
q8,1,1,1,1,1,1,1,1,0,0,...,0,0,1,1,1,1,0,0,0,0
q9,1,1,1,1,1,1,1,1,0,0,...,0,0,1,1,1,1,0,0,0,0


## 12. Metricas auxiliares

Ademas de BLEU, se calcula chrF, longitud promedio y una tasa simple de repeticion. Esto ayuda a detectar casos donde una configuracion es rapida pero genera salidas repetitivas o demasiado cortas.


In [14]:
# Utilidades numericas y valores faltantes.
import math
# Tokenizacion simple para medir repeticion.
import re
# Promedios numericos.
import numpy as np

# Metrica BLEU.
bleu_metric = evaluate.load('sacrebleu')
# Metrica chrF.
chrf_metric = evaluate.load('chrf')


# Calcula repeticion de n-gramas.
def ngram_repetition_rate(text, n=2):
    tokens = re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    if not ngrams:
        return 0.0
    repeated = len(ngrams) - len(set(ngrams))
    return repeated / len(ngrams)


def safe_mean(values):
    values = list(values)
    return float(np.mean(values)) if values else math.nan


# Agrupa metricas automaticas de traduccion.
def score_translation_outputs(predictions, references):
    bleu = bleu_metric.compute(
        predictions=predictions,
        references=[[r] for r in references],
    )['score']
    chrf = chrf_metric.compute(
        predictions=predictions,
        references=references,
    )['score']
    return {
        'BLEU': bleu,
        'chrF': chrf,
        'avg_prediction_chars': safe_mean(len(p) for p in predictions),
        'avg_reference_chars': safe_mean(len(r) for r in references),
        'avg_bigram_repetition': safe_mean(ngram_repetition_rate(p, n=2) for p in predictions),
    }


## 13. Ablation study de E2D2

Esta es la tabla central para darle peso experimental al proyecto. Se evalua E2D2 con distintas configuraciones de bloque, pasos de difusion y cache. La configuracion oficial del checkpoint WMT es `block_size=4`, `num_steps=4`, `use_cache=True`.


In [15]:
# Tamano de muestra robusta usado en el experimento final.
ROBUST_N = 30
# Limite de longitud de salida para comparar configuraciones.
MAX_NEW_TOKENS_ABLATION = 96

if len(sample_rows) < ROBUST_N:
    wmt_robust = load_dataset('wmt/wmt14', 'de-en', split=f'test[:{ROBUST_N}]')
    sample_rows_robust = [
        {'de': row['translation']['de'], 'reference_en': row['translation']['en']}
        for row in wmt_robust
    ]
else:
    sample_rows_robust = sample_rows[:ROBUST_N]

# Configuraciones E2D2 del estudio de ablacion.
ablation_grid = [
    {'config': 'E2D2 small_b2_s2_cache', 'block_size': 2, 'num_steps': 2, 'use_cache': True},
    {'config': 'E2D2 official_b4_s4_cache', 'block_size': 4, 'num_steps': 4, 'use_cache': True},
    {'config': 'E2D2 large_b8_s8_cache', 'block_size': 8, 'num_steps': 8, 'use_cache': True},
    {'config': 'E2D2 official_b4_s4_no_cache', 'block_size': 4, 'num_steps': 4, 'use_cache': False},
]

# Resultados individuales de la ablacion.
ablation_outputs = []
# Resumen por configuracion de la ablacion.
ablation_summary = []

# Recorre cada configuracion E2D2.
for cfg in ablation_grid:
    print('Running', cfg['config'])
    cfg_predictions = []
    cfg_references = []
    cfg_seconds = []
    cfg_tps = []
    cfg_generated_tokens = []

    for row_id, row in enumerate(sample_rows_robust):
        try:
            out = translate_e2d2(
                row['de'],
                max_new_tokens=MAX_NEW_TOKENS_ABLATION,
                block_size=cfg['block_size'],
                num_steps=cfg['num_steps'],
                use_cache=cfg['use_cache'],
            )
            prediction = out['prediction_en']
            cfg_predictions.append(prediction)
            cfg_references.append(row['reference_en'])
            cfg_seconds.append(out['seconds'])
            cfg_tps.append(out['tokens_per_second'])
            cfg_generated_tokens.append(out['generated_tokens'])
            status = 'ok'
            error = None
        except Exception as exc:
            prediction = ''
            status = 'error'
            error = repr(exc)

        ablation_outputs.append({
            'config': cfg['config'],
            'row_id': row_id,
            'source_de': row['de'],
            'reference_en': row['reference_en'],
            'prediction_en': prediction,
            'status': status,
            'error': error,
            'block_size': cfg['block_size'],
            'num_steps': cfg['num_steps'],
            'use_cache': cfg['use_cache'],
            'seconds': cfg_seconds[-1] if status == 'ok' else math.nan,
            'tokens_per_second': cfg_tps[-1] if status == 'ok' else math.nan,
            'generated_tokens': cfg_generated_tokens[-1] if status == 'ok' else math.nan,
            'bigram_repetition': ngram_repetition_rate(prediction, n=2),
        })

    metric_values = score_translation_outputs(cfg_predictions, cfg_references) if cfg_predictions else {}
    ablation_summary.append({
        'config': cfg['config'],
        'block_size': cfg['block_size'],
        'num_steps': cfg['num_steps'],
        'use_cache': cfg['use_cache'],
        'n_ok': len(cfg_predictions),
        'avg_seconds': safe_mean(cfg_seconds),
        'avg_tokens_per_second': safe_mean(cfg_tps),
        'avg_generated_tokens': safe_mean(cfg_generated_tokens),
        **metric_values,
    })

# Tabla resumen de la ablacion.
ablation_summary_df = pd.DataFrame(ablation_summary).sort_values('config')
ablation_outputs_df = pd.DataFrame(ablation_outputs)

display(ablation_summary_df)
# Exporta resumen E2D2 para el informe.
ablation_summary_df.to_csv('e2d2_wmt_ablation_summary.csv', index=False)
# Exporta salidas E2D2 por ejemplo.
ablation_outputs_df.to_csv('e2d2_wmt_ablation_outputs.csv', index=False)


Running E2D2 small_b2_s2_cache
Running E2D2 official_b4_s4_cache
Running E2D2 large_b8_s8_cache
Running E2D2 official_b4_s4_no_cache


,config,block_size,num_steps,use_cache,n_ok,avg_seconds,avg_tokens_per_second,avg_generated_tokens,BLEU,chrF,avg_prediction_chars,avg_reference_chars,avg_bigram_repetition
2,E2D2 large_b8_s8_cache,8,8,True,30,0.950102,71.034256,65.866667,10.129938,45.649627,260.000000,137.333333,0.240329
1,E2D2 official_b4_s4_cache,4,4,True,30,1.066114,37.235805,36.533333,17.854091,49.999465,154.433333,137.333333,0.091730
3,E2D2 official_b4_s4_no_cache,4,4,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,E2D2 small_b2_s2_cache,2,2,True,30,0.140007,15.754628,2.000000,0.000000,0.314121,1.400000,137.333333,0.033333


### Interpretacion de la ablation E2D2

La configuracion oficial `block_size=4`, `num_steps=4`, `use_cache=True` fue la mas equilibrada dentro de E2D2: BLEU 17.85, chrF 50.00 y repeticion promedio 0.092. `block_size=8` aumento el throughput, pero redujo BLEU a 10.13 y elevo la repeticion a 0.240. `block_size=2` genero salidas casi vacias. La configuracion `use_cache=False` fallo por error de dimensiones, por lo que se reporta como limitacion tecnica de esta ruta de inferencia.


## 14. Baseline autoregresivo encoder-decoder

Se compara contra un Transformer encoder-decoder autoregresivo tradicional: `Helsinki-NLP/opus-mt-de-en`. Ambos modelos se evalualan sobre la misma muestra WMT.


In [16]:
# Carga el baseline encoder-decoder autoregresivo.
from transformers import AutoModelForSeq2SeqLM

# Modelo OPUS-MT aleman-ingles usado como baseline.
BASELINE_ID = 'Helsinki-NLP/opus-mt-de-en'

# Revision exacta del baseline OPUS-MT para reproducibilidad.
BASELINE_REVISION = '1a922f3b32a8e809e17a47d4b32142d8105924e5'

# Tokenizador del baseline OPUS-MT.
baseline_tokenizer = AutoTokenizer.from_pretrained(BASELINE_ID, revision=BASELINE_REVISION)
# Carga y mueve OPUS-MT al mismo dispositivo.
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(BASELINE_ID, revision=BASELINE_REVISION).to(DEVICE)
baseline_model.eval()

baseline_params = sum(p.numel() for p in baseline_model.parameters())
print(f'Baseline: {BASELINE_ID}')
print(f'Parametros baseline: {baseline_params/1e6:.1f}M')


# Funcion de traduccion autoregresiva del baseline.
def translate_opus(german_text: str, max_new_tokens: int = 96):
    inputs = baseline_tokenizer(german_text, return_tensors='pt', truncation=True).to(DEVICE)

    # Ruta preferida: ejecucion con GPU CUDA en Colab.
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    # Inicio de medicion de tiempo.
    t0 = time.perf_counter()
    # Desactiva gradientes porque solo se hace inferencia.
    with torch.inference_mode():
        output_ids = baseline_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=1,
            do_sample=False,
        )
    if USING_XLA:
        try:
            import torch_xla.core.xla_model as xm
            xm.mark_step()
        except Exception:
            pass
    # Ruta preferida: ejecucion con GPU CUDA en Colab.
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    decoded = baseline_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    generated_tokens = int(output_ids.shape[-1])
    return {
        'prediction_en': decoded,
        'seconds': elapsed,
        'generated_tokens': generated_tokens,
        'tokens_per_second': generated_tokens / max(elapsed, 1e-9),
    }

opus_outputs = []
opus_predictions = []
opus_references = []

for row_id, row in enumerate(sample_rows_robust):
    out = translate_opus(row['de'], max_new_tokens=MAX_NEW_TOKENS_ABLATION)
    opus_predictions.append(out['prediction_en'])
    opus_references.append(row['reference_en'])
    opus_outputs.append({
        'config': 'OPUS autoregressive_encoder_decoder',
        'row_id': row_id,
        'source_de': row['de'],
        'reference_en': row['reference_en'],
        'prediction_en': out['prediction_en'],
        'seconds': out['seconds'],
        'tokens_per_second': out['tokens_per_second'],
        'generated_tokens': out['generated_tokens'],
        'bigram_repetition': ngram_repetition_rate(out['prediction_en'], n=2),
    })

opus_metrics = score_translation_outputs(opus_predictions, opus_references)
opus_summary_df = pd.DataFrame([{
    'config': 'OPUS autoregressive_encoder_decoder',
    'block_size': None,
    'num_steps': None,
    'use_cache': None,
    'n_ok': len(opus_predictions),
    'avg_seconds': safe_mean(x['seconds'] for x in opus_outputs),
    'avg_tokens_per_second': safe_mean(x['tokens_per_second'] for x in opus_outputs),
    'avg_generated_tokens': safe_mean(x['generated_tokens'] for x in opus_outputs),
    **opus_metrics,
}])

comparison_summary_df = pd.concat([
    ablation_summary_df,
    opus_summary_df,
], ignore_index=True)

display(comparison_summary_df)

opus_outputs_df = pd.DataFrame(opus_outputs)
# Exporta comparacion final E2D2 vs OPUS-MT.
comparison_summary_df.to_csv('translation_model_comparison_summary.csv', index=False)
# Exporta salidas del baseline por ejemplo.
opus_outputs_df.to_csv('opus_wmt_outputs.csv', index=False)


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Baseline: Helsinki-NLP/opus-mt-de-en
Parametros baseline: 74.4M


model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

,config,block_size,num_steps,use_cache,n_ok,avg_seconds,avg_tokens_per_second,avg_generated_tokens,BLEU,chrF,avg_prediction_chars,avg_reference_chars,avg_bigram_repetition
0,E2D2 large_b8_s8_cache,8,8,True,30,0.950102,71.034256,65.866667,10.129938,45.649627,260.000000,137.333333,0.240329
1,E2D2 official_b4_s4_cache,4,4,True,30,1.066114,37.235805,36.533333,17.854091,49.999465,154.433333,137.333333,0.091730
2,E2D2 official_b4_s4_no_cache,4,4,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E2D2 small_b2_s2_cache,2,2,True,30,0.140007,15.754628,2.000000,0.000000,0.314121,1.400000,137.333333,0.033333
4,OPUS autoregressive_encoder_decoder,None,None,None,30,0.335234,109.554263,27.633333,24.362560,54.847198,121.100000,137.333333,0.009043


### Interpretacion de la comparacion con OPUS-MT

En esta muestra, OPUS-MT supera a E2D2 en BLEU, chrF, tiempo promedio y repeticion. Este resultado no invalida E2D2 como objeto del proyecto: el objetivo de la consigna no es demostrar superioridad absoluta frente a un traductor especializado, sino implementar y analizar una arquitectura Transformer encoder-decoder basada en difusion discreta con pesos preentrenados. La comparacion permite discutir las diferencias entre decodificacion autoregresiva y denoising por bloques.


## 15. Analisis cualitativo de errores

Esta sección facilita una discusión más humana de los resultados: repeticiones, salidas cortas, traducciones literales y diferencias entre E2D2 y el baseline autoregresivo.


In [17]:
# Selecciona la mejor configuracion E2D2 para analisis cualitativo.
official_e2d2 = ablation_outputs_df[
    (ablation_outputs_df['config'] == 'E2D2 official_b4_s4_cache')
    & (ablation_outputs_df['status'] == 'ok')
].copy()

official_e2d2 = official_e2d2[[
    'row_id', 'source_de', 'reference_en', 'prediction_en', 'seconds',
    'tokens_per_second', 'bigram_repetition'
]].rename(columns={
    'prediction_en': 'e2d2_prediction',
    'seconds': 'e2d2_seconds',
    'tokens_per_second': 'e2d2_tokens_per_second',
    'bigram_repetition': 'e2d2_bigram_repetition',
})

opus_join = opus_outputs_df[[
    'row_id', 'prediction_en', 'seconds', 'tokens_per_second', 'bigram_repetition'
]].rename(columns={
    'prediction_en': 'opus_prediction',
    'seconds': 'opus_seconds',
    'tokens_per_second': 'opus_tokens_per_second',
    'bigram_repetition': 'opus_bigram_repetition',
})

# Une E2D2 y OPUS por ejemplo.
qualitative_df = official_e2d2.merge(opus_join, on='row_id', how='inner')


# Diagnostica patrones simples de error.
def diagnose_row(row):
    notes = []
    ref_len = max(len(row['reference_en']), 1)
    e2d2_len_ratio = len(row['e2d2_prediction']) / ref_len
    if row['e2d2_bigram_repetition'] > 0.12:
        notes.append('E2D2 con repeticion')
    if e2d2_len_ratio < 0.55:
        notes.append('E2D2 salida corta')
    if e2d2_len_ratio > 1.45:
        notes.append('E2D2 salida larga')
    if row['opus_bigram_repetition'] > 0.12:
        notes.append('OPUS con repeticion')
    if not notes:
        notes.append('comparar fidelidad y fluidez')
    return '; '.join(notes)

qualitative_df['nota_analisis'] = qualitative_df.apply(diagnose_row, axis=1)

display(qualitative_df[[
    'row_id', 'source_de', 'reference_en', 'e2d2_prediction', 'opus_prediction',
    'e2d2_bigram_repetition', 'opus_bigram_repetition', 'nota_analisis'
]].head(12))

# Exporta analisis cualitativo.
qualitative_df.to_csv('qualitative_error_analysis.csv', index=False)


,row_id,source_de,reference_en,e2d2_prediction,opus_prediction,e2d2_bigram_repetition,opus_bigram_repetition,nota_analisis
0,0,Gutach: Noch mehr Sicherheit für Fußgänger,Gutach: Increased safety for pedestrians,Well: more safety for pedestrians.,Gutach: More safety for pedestrians,0.000000,0.0,comparar fidelidad y fluidez
1,1,Sie stehen keine 100 Meter voneinander entfern...,They are not even 100 metres apart: On Tuesday...,They are not 100 metres away: On Tuesday the n...,They are not 100 meters apart: On Tuesday the ...,0.000000,0.0,comparar fidelidad y fluidez
2,2,Zwei Anlagen so nah beieinander: Absicht oder ...,Two sets of lights so close to one another: in...,Two plants so close to each other: intention o...,Two plants so close together: intention or shi...,0.000000,0.0,comparar fidelidad y fluidez
3,3,Diese Frage hat Gutachs Bürgermeister gestern ...,"Yesterday, Gutacht's Mayor gave a clear answer...",Gutach Mayor gave a clear answer to this quest...,Gutach's mayor answered that question clearly ...,0.272727,0.0,E2D2 con repeticion; E2D2 salida larga
4,4,"""Die Rathausampel ist damals installiert worde...","""At the time, the Town Hall traffic lights wer...","""The Town Hall lamp was installed at that time...","""The town hall lamp was installed at the time ...",0.000000,0.0,comparar fidelidad y fluidez
5,5,Die Kluser-Ampel sichere sowohl Radfahrer als ...,"The Kluser lights protect cyclists, as well as...",The Kluser lights protect both cyclists and bu...,The Kluser traffic lights are safe for cyclist...,0.187500,0.0,E2D2 con repeticion; E2D2 salida larga
6,6,Die gestern offiziell in Betrieb genommene Anl...,"The system, which officially became operationa...",The facility officially opened yesterday was i...,The plant officially commissioned yesterday wa...,0.178571,0.0,E2D2 con repeticion; E2D2 salida larga
7,7,"Wir haben das Museum, zwei Kirchen, Kurpark, d...","We have the museum, two churches, the spa gard...","We have a museum, two churches, spa park, bus ...","We have the museum, two churches, spa park, bu...",0.000000,0.0,comparar fidelidad y fluidez
8,8,"""Bei dem hohen Verkehrs- und Fußgängeraufkomme...","""At times of high road and pedestrian traffic,...","""The high traffic and pedestrian volume had to...","""With the high traffic and pedestrian traffic,...",0.000000,0.0,comparar fidelidad y fluidez
9,9,Dies bestätigt auch Peter Arnold vom Landratsa...,This was also confirmed by Peter Arnold from t...,This is also confirmed by Peter Arnold of the ...,This is confirmed by Peter Arnold of the Offen...,0.000000,0.0,comparar fidelidad y fluidez


### Interpretacion cualitativa

El analisis cualitativo confirma lo observado en las metricas: E2D2 puede producir traducciones razonables, pero en algunos ejemplos cae en repeticiones o continuaciones no justificadas. El caso `Pro Fahrtrichtung gibt es drei Lichtanlagen` es ilustrativo: E2D2 traduce el inicio correctamente, pero luego repite una frase ajena sobre un automovil de 100 m; OPUS-MT genera una salida breve y fiel. Esta evidencia justifica incluir métricas de repeticion ademas de BLEU y chrF.


## 16.1. Evaluación extendida con 150 ejemplos

La evaluación inicial de 30 ejemplos permite iterar rápido, pero conviene ampliar la muestra sin intentar reproducir todo el test set. Esta sección evalúa 150 ejemplos con las dos configuraciones más importantes:

1. E2D2 oficial: `block_size=4`, `num_steps=4`, `use_cache=True`.
2. OPUS-MT autoregresivo: baseline encoder-decoder tradicional.

No se repiten todas las ablaciones en 150 ejemplos porque su objetivo es estudiar sensibilidad de parámetros, no reemplazar la evaluación principal. Esta celda guarda resultados parciales cada 25 ejemplos para reducir riesgo de pérdida si Colab se desconecta.


In [18]:
# Tamano de la evaluacion extendida. Es mayor que 30, pero sigue siendo viable en Colab.
EXTENDED_N = 150

# Longitud maxima de salida para mantener comparabilidad con la evaluacion anterior.
MAX_NEW_TOKENS_EXTENDED = 96

# Descarga 150 ejemplos del test set WMT14 de-en.
wmt_extended = load_dataset('wmt/wmt14', 'de-en', split=f'test[:{EXTENDED_N}]')

# Convierte la muestra extendida a una lista simple de fuente y referencia.
extended_rows = [
    {'de': row['translation']['de'], 'reference_en': row['translation']['en']}
    for row in wmt_extended
]

# Acumuladores de resultados por ejemplo.
extended_outputs = []

# Evalua E2D2 oficial y OPUS-MT sobre la misma muestra extendida.
for row_id, row in enumerate(extended_rows):
    # Ejecuta E2D2 con la configuracion oficial del checkpoint.
    e2d2_out = translate_e2d2(
        row['de'],
        max_new_tokens=MAX_NEW_TOKENS_EXTENDED,
        block_size=4,
        num_steps=4,
        use_cache=True,
    )

    # Ejecuta OPUS-MT como baseline autoregresivo.
    opus_out = translate_opus(row['de'], max_new_tokens=MAX_NEW_TOKENS_EXTENDED)

    # Guarda ambos resultados en formato largo para facilitar calculo de metricas.
    extended_outputs.append({
        'config': 'E2D2 official_b4_s4_cache',
        'row_id': row_id,
        'source_de': row['de'],
        'reference_en': row['reference_en'],
        'prediction_en': e2d2_out['prediction_en'],
        'seconds': e2d2_out['seconds'],
        'tokens_per_second': e2d2_out['tokens_per_second'],
        'generated_tokens': e2d2_out['generated_tokens'],
        'bigram_repetition': ngram_repetition_rate(e2d2_out['prediction_en'], n=2),
    })
    extended_outputs.append({
        'config': 'OPUS autoregressive_encoder_decoder',
        'row_id': row_id,
        'source_de': row['de'],
        'reference_en': row['reference_en'],
        'prediction_en': opus_out['prediction_en'],
        'seconds': opus_out['seconds'],
        'tokens_per_second': opus_out['tokens_per_second'],
        'generated_tokens': opus_out['generated_tokens'],
        'bigram_repetition': ngram_repetition_rate(opus_out['prediction_en'], n=2),
    })

    # Guarda progreso cada 25 ejemplos para no perder resultados si Colab se interrumpe.
    if (row_id + 1) % 25 == 0:
        pd.DataFrame(extended_outputs).to_csv('extended_150_outputs_partial.csv', index=False)
        print(f'Progreso guardado: {row_id + 1}/{EXTENDED_N}')

# DataFrame completo de salidas extendidas.
extended_outputs_df = pd.DataFrame(extended_outputs)
extended_outputs_df.to_csv('extended_150_outputs.csv', index=False)

# Resume metricas por configuracion.
extended_summary = []
for cfg_name, group in extended_outputs_df.groupby('config'):
    predictions = group['prediction_en'].tolist()
    references = group['reference_en'].tolist()
    metrics = score_translation_outputs(predictions, references)
    extended_summary.append({
        'config': cfg_name,
        'n': len(group),
        'avg_seconds': group['seconds'].mean(),
        'avg_tokens_per_second': group['tokens_per_second'].mean(),
        'avg_generated_tokens': group['generated_tokens'].mean(),
        'avg_bigram_repetition': group['bigram_repetition'].mean(),
        **metrics,
    })

extended_summary_df = pd.DataFrame(extended_summary)
extended_summary_df.to_csv('extended_150_summary.csv', index=False)
display(extended_summary_df)


Progreso guardado: 25/150
Progreso guardado: 50/150
Progreso guardado: 75/150
Progreso guardado: 100/150
Progreso guardado: 125/150
Progreso guardado: 150/150


,config,n,avg_seconds,avg_tokens_per_second,avg_generated_tokens,avg_bigram_repetition,BLEU,chrF,avg_prediction_chars,avg_reference_chars
0,E2D2 official_b4_s4_cache,150,0.70555,50.319964,32.506667,0.076863,16.527933,49.664514,149.006667,118.12
1,OPUS autoregressive_encoder_decoder,150,0.18943,144.682039,24.613333,0.004980,25.017465,54.091530,109.013333,118.12


## 16. Artefactos generados

Esta sección se genera para poder visualizar y analizar los resultados generados. Apartir de correr el código se generan los archivos .CSV con la información.

In [19]:
# Permite verificar archivos exportados.
import os

# Lista de artefactos generados por el notebook.
artifact_files = [
    'e2d2_wmt_ablation_summary.csv',
    'e2d2_wmt_ablation_outputs.csv',
    'translation_model_comparison_summary.csv',
    'opus_wmt_outputs.csv',
    'qualitative_error_analysis.csv',
]

for file_name in artifact_files:
    if os.path.exists(file_name):
        print(file_name, '-', round(os.path.getsize(file_name) / 1024, 2), 'KB')
    else:
        print(file_name, '- no generado todavia')


e2d2_wmt_ablation_summary.csv - 0.7 KB
e2d2_wmt_ablation_outputs.csv - 56.84 KB
translation_model_comparison_summary.csv - 0.88 KB
opus_wmt_outputs.csv - 14.18 KB
qualitative_error_analysis.csv - 19.97 KB
